In [5]:
import pandas as pd
import numpy as np
import re

from gensim.models import Word2Vec
from gensim.utils import simple_preprocess

In [2]:
data = pd.read_csv(r'D:\Downloads\LP IV Updated Datasets\LP IV Updated Datasets\CBOW\twitter_dataset.csv')

In [3]:
print("Dataset Shape: ", data.shape)
print(data.head())

Dataset Shape:  (10000, 6)
   Tweet_ID        Username  \
0         1         julie81   
1         2   richardhester   
2         3  williamsjoseph   
3         4     danielsmary   
4         5      carlwarren   

                                                Text  Retweets  Likes  \
0  Party least receive say or single. Prevent pre...         2     25   
1  Hotel still Congress may member staff. Media d...        35     29   
2  Nice be her debate industry that year. Film wh...        51     25   
3  Laugh explain situation career occur serious. ...        37     18   
4  Involve sense former often approach government...        27     80   

             Timestamp  
0  2023-01-30 11:00:51  
1  2023-01-02 22:45:58  
2  2023-01-18 11:25:19  
3  2023-04-10 22:06:29  
4  2023-01-24 07:12:21  


In [4]:
tweets = data['Text'].dropna().astype(str)

In [6]:
def preprocess(text):
    text = text.lower()

    text = re.sub(r'http\S+ | www\S+', '', text)

    text = re.sub(r'@\w+', '', text)

    text = re.sub(r'#','', text)

    tokens = simple_preprocess(text, deacc=True)

    return tokens

In [7]:
tokenized_tweets = [preprocess(tweet) for tweet in tweets]

tokenized_tweets = [t for t in tokenized_tweets if len(t)>0]

In [8]:
print("Tokenized tweets: ", tokenized_tweets[:3])

Tokenized tweets:  [['party', 'least', 'receive', 'say', 'or', 'single', 'prevent', 'prevent', 'husband', 'affect', 'may', 'himself', 'cup', 'style', 'evening', 'protect', 'effect', 'another', 'themselves', 'stage', 'perform', 'possible', 'try', 'tax', 'share', 'style', 'television', 'with', 'successful', 'much', 'sell', 'development', 'economy', 'effect'], ['hotel', 'still', 'congress', 'may', 'member', 'staff', 'media', 'draw', 'buy', 'fly', 'identify', 'on', 'another', 'turn', 'minute', 'would', 'local', 'subject', 'way', 'believe', 'which', 'question', 'some', 'message', 'own', 'all', 'imagine', 'join', 'agency', 'indicate'], ['nice', 'be', 'her', 'debate', 'industry', 'that', 'year', 'film', 'where', 'generation', 'push', 'discover', 'partner', 'level', 'nearly', 'money', 'store', 'style', 'may', 'enjoy', 'kid', 'discuss', 'blue', 'save', 'model', 'another', 'about', 'along', 'everybody', 'especially', 'dinner', 'character', 'yard']]


In [9]:
model = Word2Vec(
    sentences=tokenized_tweets,
    min_count=2,
    window=5,
    sg=0,
    vector_size=100,
    workers=4,
    epochs=20
)

In [10]:
print("Vocabulary size: ", len(model.wv))

print("\nFirst 10 words: ")
print(model.wv.index_to_key[:10])

Vocabulary size:  969

First 10 words: 
['hard', 'tax', 'out', 'add', 'when', 'my', 'high', 'maybe', 'food', 'yourself']


In [11]:
word = model.wv.index_to_key[0]

embedding = model.wv[word]

print("Selected word: ", word)
print("Word Embedding: ")
print(embedding)

Selected word:  hard
Word Embedding: 
[-0.31465995 -0.30000266 -0.18102685  0.34229589 -0.55113864 -0.68399394
  0.32800052  0.32843673 -0.11560737  0.4057345  -0.06901456  0.26751393
 -0.01091336  0.20810844 -0.06443886  0.06880566  0.29354545  0.18044311
  0.2914215  -0.94532734 -0.0551644   0.06687132 -0.00145922  0.22927265
  0.46498936  0.18265845 -0.6296213   0.30160472  0.18083826  0.31118488
  0.03601561 -0.34349844  0.58631146  0.17271902 -0.4060566   0.30756512
 -0.7938439  -0.2731223   0.03779911 -0.31268737 -0.02958354  0.30347848
 -0.31720555  0.41233876  0.13251647  0.33152425 -0.10986389  0.42065564
  0.20929484  0.02702279 -0.3387429  -0.64625776  0.03902918  0.56089044
  0.03435786 -0.01239119 -0.34855717 -0.23851998 -0.2531396   0.5848352
  0.9109624   0.6493783  -0.3491271  -0.25133857 -0.501247   -0.02795742
 -0.31073818  0.11395995  0.12070171 -0.18093431 -0.3683523   0.13544722
  0.19080798 -0.08660682  0.27592444  0.3918071   0.27672482  0.22833796
  0.32954997 -

In [12]:
similar_word = model.wv.most_similar(word, topn=5)
print("\nWords Similar to", word)

for w, score in similar_word:
    print(w, ":", score)


Words Similar to hard
late : 0.46812504529953003
very : 0.41500425338745117
report : 0.39965683221817017
cut : 0.38393428921699524
day : 0.38123539090156555


In [13]:
word1 = model.wv.index_to_key[0]
word2 = model.wv.index_to_key[1]

similarity = model.wv.similarity(word1, word2)

print("Similarity between ", word1, "and", word2, ": ", similarity)


Similarity between  hard and tax :  0.21277711


In [14]:
model.save("cbow_twitter.model")